In [ ]:
import pandas as pd
import commons as c
import plotly.express as px
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
csv_normal_path = 'results_noise_analysis/dataframes/all_data.csv'
df = pd.read_csv(csv_normal_path)

# KDE plot ideal distance vs noisy distance

In [ ]:
custom_order = ['Trace', 'Fidelity', 'Hellinger', 'Jensen-Shannon', 'Expectation Values']

# Convert the column to a categorical type with that order
cat_type = pd.CategoricalDtype(categories=custom_order, ordered=True)
df['metric_full'] = df['metric_full'].astype(cat_type)

# Sort by that custom order
df = df.sort_values('metric_full')
df['hardware'] = df['hardware'].str.capitalize()

df_filtered = df[df['threshold']=='I']

# Unique values for layout
hardware_types = df_filtered["hardware"].unique()
metrics = df_filtered["metric_full"].unique()
n_rows = len(hardware_types)
n_cols = len(metrics)

# Set limits if you want them consistent
min_val = min(df_filtered["ideal_distance"].min(), df_filtered["noisy_distance"].min())
max_val = max(df_filtered["ideal_distance"].max(), df_filtered["noisy_distance"].max())

# Create subplots
fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, 4 * n_rows), sharex=True, sharey=True)
axes = axes.reshape(n_rows, n_cols)  # Ensure 2D shape

# Plotting loop
for i, hardware in enumerate(hardware_types):
    for j, metric in enumerate(metrics):
        ax = axes[i, j]
        subset = df_filtered[(df_filtered["hardware"] == hardware) & (df_filtered["metric_full"] == metric)]

        if subset.empty:
            ax.axis("off")
            continue

        sns.kdeplot(
            x=subset["ideal_distance"],
            y=subset["noisy_distance"],
            fill=True,
            cmap=sns.color_palette("ch:start=.2,rot=-.3", as_cmap=True),
            ax=ax,
        )
        
        ax.plot([min_val, max_val], [min_val, max_val], linestyle="dashed", color="black")
        ax.set_xlim(0, 1)
        ax.set_ylim(0, 1)
        ax.tick_params(axis='both', labelsize=14)
        if i == 0:
            ax.set_title(metric, fontsize=24)

        ax.set_ylabel(f"{hardware}", fontsize=26, labelpad=30)

        # Remove individual axis labels
        ax.set_xlabel("")
        # ax.set_ylabel("")

# Shared axis labels
fig.text(0.52, 0.01, 'Noiseless Distance', ha='center', fontsize=26)
fig.text(0.01, 0.52, 'Noisy Distance', va='center', rotation='vertical', fontsize=26)

# Main title
# fig.suptitle("KDE of Ideal vs. Noisy Distance by Hardware and Metric", fontsize=16)

# Layout adjustment
plt.tight_layout(rect=[0.03, 0.05, 1, 0.95])
plt.subplots_adjust(hspace=0.12, wspace=0.12)  # Reduce these values for tighter plots

plt.show()

# Boxplot of the distance between Ideal and Noisy executions 

In [ ]:

df = df.melt(
    id_vars=['Algorithm', 'Qubits_number', 'hardware', 'nature', 'metric_full', 'hardware_named'],
    value_vars=['ideal_distance', 'noisy_distance'],
    value_name='distance'
)

# Replace 'hardware' with 'ideal' where the row comes from 'ideal_distance'
df.loc[df['variable'] == 'ideal_distance', 'hardware'] = 'ideal'
df.loc[df['variable'] == 'ideal_distance', 'hardware_named'] = 'Noiseless'

# Drop the 'variable' column as it's no longer needed
df = df.drop(columns=['variable'])

In [ ]:
fig = px.box(
    df,
    x='metric_full',
    y='distance',
    color= 'hardware_named',
    labels={"metric_full": "Metric", 'distance': "Distance", 'hardware_named': "Simulator"},
    points=False,
    boxmode="group",
    color_discrete_map=c.color_map
)

# Save the figure
output_folder = 'results/RQ0/'
c.setup_layout_and_save(fig, output_folder, f'visu_distance', yaxis_range=[0, 1])
